# Part 5: Putting It All Together

This is the complete, self-contained notebook — everything from Parts 1–4 assembled in one place. Train a GPT on Shakespeare, watch the loss curve, generate text, and run experiments.

> **Tip for Colab**: Go to **Runtime → Change runtime type → GPU (T4)** before running. Training the Medium config takes ~15 min on a T4 GPU, vs hours on CPU.

## Setup

In [ ]:
!pip install -q torch tqdm matplotlib

# Download the Shakespeare dataset
import urllib.request, os
os.makedirs("data", exist_ok=True)
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt",
    "data/shakespeare.txt"
)
print(f"Dataset size: {os.path.getsize('data/shakespeare.txt') / 1e6:.1f} MB")

## The Model

In [ ]:
import torch
import torch.nn as nn
from dataclasses import dataclass
import math, json
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

@dataclass
class GPTConfig:
    vocab_size: int = 65
    block_size: int = 256
    n_layer: int = 6
    n_head: int = 6
    n_embd: int = 384

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)
        y = torch.nn.functional.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        return self.c_proj(self.gelu(self.c_fc(x)))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)
        x = self.transformer.wte(idx) + self.transformer.wpe(pos)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1)
            )
        return logits, loss

print("Model classes ready.")

## Data Loading, Training, and Generation

In [ ]:
def get_device():
    if torch.backends.mps.is_available():
        return torch.device("mps")
    elif torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

def load_data(filepath, block_size, batch_size, device):
    with open(filepath, "r") as f:
        text = f.read()
    chars = sorted(set(text))
    vocab_size = len(chars)
    stoi = {c: i for i, c in enumerate(chars)}
    itos = {i: c for c, i in stoi.items()}
    tokens = torch.tensor([stoi[c] for c in text], dtype=torch.long)
    print(f"Dataset: {len(tokens):,} chars, vocab size: {vocab_size}")
    def get_batch(split_tokens):
        ix = torch.randint(len(split_tokens) - block_size - 1, (batch_size,))
        x = torch.stack([split_tokens[i:i + block_size] for i in ix]).to(device)
        y = torch.stack([split_tokens[i + 1:i + block_size + 1] for i in ix]).to(device)
        return x, y
    n = int(0.9 * len(tokens))
    return lambda: get_batch(tokens[:n]), lambda: get_batch(tokens[n:]), vocab_size, stoi, itos

def get_lr(step, warmup_steps, max_steps, max_lr, min_lr):
    if step < warmup_steps:
        return max_lr * (step + 1) / warmup_steps
    if step >= max_steps:
        return min_lr
    progress = (step - warmup_steps) / (max_steps - warmup_steps)
    return min_lr + 0.5 * (max_lr - min_lr) * (1 + math.cos(math.pi * progress))

@torch.no_grad()
def generate(model, prompt, stoi, itos, max_new_tokens=200, temperature=0.8, top_k=40):
    device = next(model.parameters()).device
    tokens = [stoi[c] for c in prompt if c in stoi]
    idx = torch.tensor([tokens], dtype=torch.long, device=device)
    model.eval()
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -model.config.block_size:]
        logits, _ = model(idx_cond)
        logits = logits[:, -1, :] / temperature
        if top_k > 0:
            values, _ = torch.topk(logits, top_k)
            logits[logits < values[:, -1:]] = float("-inf")
        probs = torch.softmax(logits, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_token], dim=1)
    return "".join([itos[i] for i in idx[0].tolist()])

def train(data_path, max_steps=5000, batch_size=64,
          n_layer=6, n_head=6, n_embd=384, block_size=256,
          checkpoint_name="checkpoint_final"):
    device = get_device()
    print(f"Device: {device}")

    get_train_batch, get_val_batch, vocab_size, stoi, itos = load_data(
        data_path, block_size, batch_size, device
    )
    config = GPTConfig(vocab_size=vocab_size, block_size=block_size,
                       n_layer=n_layer, n_head=n_head, n_embd=n_embd)
    model = GPT(config).to(device)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"Model: {n_layer}L/{n_head}H/{n_embd}D — {n_params/1e6:.1f}M params")

    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    max_lr, min_lr, warmup_steps = 1e-3, 1e-4, 100
    loss_log = {"steps": [], "train": [], "val_steps": [], "val": []}

    pbar = tqdm(range(max_steps), desc="Training")
    for step in pbar:
        if step % 100 == 0:
            model.eval()
            with torch.no_grad():
                val_loss = sum(
                    model(*get_val_batch())[1].item() for _ in range(20)
                ) / 20
                tqdm.write(f"Step {step:5d} | val loss: {val_loss:.4f}")
                loss_log["val_steps"].append(step)
                loss_log["val"].append(val_loss)
            model.train()

        lr = get_lr(step, warmup_steps, max_steps, max_lr, min_lr)
        for g in optimizer.param_groups:
            g["lr"] = lr

        x, y = get_train_batch()
        _, loss = model(x, y)
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{lr:.2e}")
        loss_log["steps"].append(step)
        loss_log["train"].append(loss.item())

        if step > 0 and step % 500 == 0:
            model.eval()
            sample = generate(model, "To be or not", stoi, itos,
                              max_new_tokens=100, temperature=0.8)
            tqdm.write(f"\n--- Step {step} sample ---\n{sample}\n---\n")
            model.train()

        if step > 0 and step % 1000 == 0:
            torch.save({
                "step": step, "model_state_dict": model.state_dict(),
                "config": config, "stoi": stoi, "itos": itos,
            }, f"{checkpoint_name}_{step}.pt")

    torch.save({
        "step": max_steps, "model_state_dict": model.state_dict(),
        "config": config, "stoi": stoi, "itos": itos,
    }, f"{checkpoint_name}.pt")

    log_path = f"{checkpoint_name}_loss_log.json"
    with open(log_path, "w") as f:
        json.dump(loss_log, f)

    print(f"\nDone. Saved {checkpoint_name}.pt and {log_path}")
    return model, stoi, itos, loss_log

print("All functions defined.")

## Train on Shakespeare

Pick a config based on your hardware:

| Config | Params | Expected Time (T4 GPU) |
|--------|--------|------------------------|
| Tiny   | ~0.5M  | ~2 min                 |
| Small  | ~4M    | ~7 min                 |
| **Medium** | **~10M** | **~15 min**        |

In [ ]:
# Choose your config:

# Tiny — fast, good for testing
# model, stoi, itos, loss_log = train("data/shakespeare.txt", n_layer=2, n_head=2, n_embd=128, max_steps=2000, checkpoint_name="tiny")

# Small
# model, stoi, itos, loss_log = train("data/shakespeare.txt", n_layer=4, n_head=4, n_embd=256, max_steps=5000, checkpoint_name="small")

# Medium (default)
model, stoi, itos, loss_log = train("data/shakespeare.txt", checkpoint_name="medium")

## Plot Loss Curves

Watch for the gap between train and val loss. When val loss starts rising while train loss keeps falling, the model is **overfitting** — memorizing training data instead of learning general patterns.

In [ ]:
def plot_loss(loss_log, title="Loss Curves"):
    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(loss_log["steps"], loss_log["train"], alpha=0.3, label="Train loss")
    ax.plot(loss_log["val_steps"], loss_log["val"], color="red", linewidth=2, label="Val loss")

    best_val_idx = loss_log["val"].index(min(loss_log["val"]))
    best_step = loss_log["val_steps"][best_val_idx]
    best_val = loss_log["val"][best_val_idx]
    ax.axvline(best_step, color='green', linestyle='--',
               label=f'Best val loss ({best_val:.3f}) at step {best_step}')

    ax.set_xlabel("Step")
    ax.set_ylabel("Loss")
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"Best val loss: {best_val:.4f} at step {best_step}")
    print(f"Final val loss: {loss_log['val'][-1]:.4f}")

plot_loss(loss_log, title="Medium Config — Shakespeare")

## Generate Text

In [ ]:
# Generate with different prompts
prompts = [
    "To be or not to be",
    "ROMEO:\n",
    "Shall I compare thee",
]

for prompt in prompts:
    torch.manual_seed(42)
    output = generate(model, prompt, stoi, itos,
                      max_new_tokens=200, temperature=0.8, top_k=40)
    print(f"\n=== Prompt: {repr(prompt)} ===")
    print(output)
    print()

## Experiments

Now that the pipeline works, try these to build intuition.

### Experiment 1: Model Size vs. Quality

Train Tiny and Small configs and compare their val losses and output quality.

In [ ]:
# Uncomment to run all configs and compare

# _, _, _, tiny_log = train("data/shakespeare.txt",
#     n_layer=2, n_head=2, n_embd=128, max_steps=5000, checkpoint_name="tiny")
# _, _, _, small_log = train("data/shakespeare.txt",
#     n_layer=4, n_head=4, n_embd=256, max_steps=5000, checkpoint_name="small")

# Then plot all three:
# for name, log in [("Tiny (~0.5M)", tiny_log), ("Small (~4M)", small_log), ("Medium (~10M)", loss_log)]:
#     plot_loss(log, title=name)

### Experiment 2: Context Length

Train with `block_size=128` vs `block_size=512`. Longer context lets the model capture full stanzas and rhyme schemes, but uses more memory (reduce `batch_size` to compensate).

In [ ]:
# Short context
# _, _, _, log_128 = train("data/shakespeare.txt", block_size=128, batch_size=128, checkpoint_name="ctx128")

# Long context (reduce batch_size to fit in memory)
# _, _, _, log_512 = train("data/shakespeare.txt", block_size=512, batch_size=32, checkpoint_name="ctx512")

### Experiment 3: Learning Rate

Try `3e-4` (conservative), `1e-3` (default), `3e-3` (aggressive). The right LR depends on your model size and data.

In [ ]:
# Modify get_lr's max_lr inside train() to try different values,
# or pass it as a parameter after extending the train() function.

## What to Look For in Loss Curves

| Pattern | Meaning |
|---------|----------|
| Train loss not decreasing | Learning rate too low, or a bug |
| Train ↓ but val ↑ | Overfitting — more data or smaller model |
| Loss spikes | Reduce learning rate or check gradient clipping |
| Loss plateaus | Model has learned what it can — needs more data or larger model |

**Next: [Part 6 — Competition](06-competition.ipynb)**